# Carga y Consolidación de Datos
## Tesis: Benchmarking Explainable Gradient Boosting and Tabular Deep Learning  
## for Predicting Satisfaction with Democracy in Latin America (1995–2024)

**Autor:** Mario Patricio Porras Martínez  
**Universidad:** Yachay Tech — Maestría en Inteligencia Artificial  

---

### Propósito de este notebook

Este notebook implementa el **pipeline de adquisición y consolidación de datos** correspondiente
a la primera etapa de la metodología de investigación. Su función es:

1. Leer los 25 archivos `.dta` de Latinobarómetro (1995–2024), extraer las variables
   seleccionadas para la tesis y consolidarlas en un único DataFrame longitudinal.
2. Exportar el DataFrame consolidado como `data/latinobarometro.csv`.
3. Leer el archivo `V-Dem-CY-Core-v16.csv`, filtrar por los países del estudio
   y el período 1995–2024, y exportar el resultado como `data/v-dem.csv`.

### Estructura de directorios esperada

```
proyecto/
├── data/
│   ├── F00018833-Latinobarometro_Serie_de_Tiempo_1995_2024.xlsx   ← diccionario de mapeo
│   ├── raw_latinobarometro/
│   │   ├── Latinobarometro_1995.dta
│   │   ├── Latinobarometro_1996.dta
│   │   │   ... (25 archivos)
│   │   └── Latinobarometro_2024.dta
│   ├── raw_v-dem/
│   │   └── V-Dem-CY-Core-v16.csv
│   └── variables/
│       └── latinobarometro_variables.csv
└── notebooks/
    └── 01_carga_datos.ipynb   ← este archivo
```

### Variables seleccionadas

Las variables fueron definidas y justificadas en el documento `variables_seleccionadas_metodologia.md`.
El mapeo entre los códigos estandarizados del diccionario maestro y los nombres originales
de cada ola se construye automáticamente a partir del archivo Excel de series de tiempo.

---

> **Nota de reproducibilidad:** Todas las semillas aleatorias están fijadas en `42`.
> Las versiones exactas de las librerías utilizadas se registran al final del notebook.

## 0. Instalación de dependencias

Ejecutar esta celda solo la primera vez o cuando se configure un nuevo entorno virtual.

In [40]:
# =============================================================================
# CELDA 0 — Instalación de dependencias
# Ejecutar una sola vez al configurar el entorno virtual (.venv)
# =============================================================================

import subprocess, sys

paquetes = [
    "pandas>=2.1.0",
    "pyreadstat>=1.2.6",   # Lectura de archivos .dta (Stata) y .sav (SPSS)
    "openpyxl>=3.1.2",     # Lectura del Excel de series de tiempo
    "numpy>=1.26.0",
    "tqdm>=4.66.0",        # Barra de progreso para la carga de archivos
]

for paquete in paquetes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", paquete, "-q"])

print("✓ Todas las dependencias instaladas correctamente.")

✓ Todas las dependencias instaladas correctamente.


## 1. Importaciones y configuración global

In [ ]:
# =============================================================================
# CELDA 1 — Importaciones y configuración global
# =============================================================================

import os
import gc
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Configuración de rutas ---------------------------------------------------
RAIZ_PROYECTO        = Path("..")
CARPETA_LB           = RAIZ_PROYECTO / "data" / "raw_latinobarometro"
CARPETA_VDEM         = RAIZ_PROYECTO / "data" / "raw_v-dem"
CARPETA_DATA         = RAIZ_PROYECTO / "data"
ARCHIVO_VARIABLES_LB = CARPETA_DATA / "variables" / "latinobarometro_variables.csv"
ARCHIVO_VDEM         = CARPETA_VDEM / "V-Dem-CY-Core-v16.csv"
SALIDA_LB            = CARPETA_DATA / "loaded" / "latinobarometro.csv"
SALIDA_VDEM          = CARPETA_DATA / "loaded" / "v-dem.csv"

# --- Verificación de existencia de rutas críticas ----------------------------
rutas_criticas = {
    "Carpeta Latinobarómetro"       : CARPETA_LB,
    "Carpeta V-Dem"                 : CARPETA_VDEM,
    "CSV Variables Latinobarómetro" : ARCHIVO_VARIABLES_LB,
    "CSV V-Dem CY-Core"             : ARCHIVO_VDEM,
}

errores = []
for nombre, ruta in rutas_criticas.items():
    estado = "✓" if ruta.exists() else "✗"
    if not ruta.exists():
        errores.append(nombre)
    print(f"  {estado}  {nombre}: {ruta}")

if errores:
    raise FileNotFoundError(
        f"No se encontraron las siguientes rutas: {errores}. "
        "Verificar la estructura de directorios."
    )

print("\n✓ Todas las rutas críticas verificadas correctamente.")

  ✓  Carpeta Latinobarómetro: ..\data\latinobarometro
  ✓  Carpeta V-Dem: ..\data\v-dem
  ✓  CSV Variables Latinobarómetro: ..\data\latinobarometro\latinobarometro_variables.csv
  ✓  CSV V-Dem CY-Core: ..\data\v-dem\V-Dem-CY-Core-v16.csv

✓ Todas las rutas críticas verificadas correctamente.


## 2. Construcción del diccionario de mapeo variable ↔ ola

El diccionario de mapeo traduce los **códigos estandarizados** del diccionario maestro
(ej. `A_003_031`) a los **nombres originales de columna** de cada archivo `.dta`
(ej. `p21` en LAT1995, `P11ST.A` en LAT2010).  

Este mapeo se construye automáticamente desde el archivo Excel de series de tiempo,
garantizando que cualquier actualización del diccionario maestro se refleje
automáticamente sin necesidad de modificar el código.

In [42]:
# =============================================================================
# CELDA 2 — Construcción del diccionario de mapeo desde el Excel
# =============================================================================

# Variables seleccionadas para la tesis (códigos estandarizados del diccionario maestro)
# Organizadas por bloque temático según la metodología de investigación
VARIABLES_SELECCIONADAS = [
    # --- Variable dependiente ---
    "A_003_031",   # Satisfacción con la democracia (TARGET)

    # --- Apoyo al régimen democrático ---
    "A_001_001",   # Apoyo a la democracia
    "A_003_021",   # Escala de desarrollo de democracia

    # --- Confianza institucional ---
    "H_002_011",   # Confianza: Congreso
    "H_002_031",   # Confianza: Gobierno
    "H_002_041",   # Confianza: Poder Judicial
    "H_002_101",   # Confianza: Iglesia
    "H_002_111",   # Confianza: Policía
    "H_002_131",   # Confianza: Televisión
    "H_002_161",   # Confianza: Fuerzas Armadas
    "H_002_241",   # Confianza: Partidos Políticos
    "H_001_011",   # Confianza interpersonal

    # --- Evaluación económica ---
    "D_001_001",   # Situación económica actual del país
    "D_001_021",   # Situación económica del país vs. hace 12 meses
    "D_001_041",   # Situación económica futura del país
    "D_001_061",   # Situación económica personal actual
    "D_001_091",   # Situación económica futura personal
    "D_001_131",   # Ingreso subjetivo
    "C_003_003_011", # Preocupación por perder el trabajo
    "C_006_003_011", # Justicia en la distribución del ingreso

    # --- Percepción política y social ---
    "B_001_101",   # País para todos o para los poderosos
    "B_006_061",   # Aprobación gestión del gobierno
    "A_007_071",   # Escala Izquierda-Derecha
    "A_007_001",   # Interés en política
    "C_001_031",   # Problema más importante del país

    # --- Corrupción y seguridad ---
    "G_002_011",   # Conocimiento directo de acto de corrupción
    "G_005_001",   # Progreso en reducción de corrupción
    "I_001_001",   # Victimización por delito

    # --- Sociodemográfico ---
    "S_001",       # Sexo del entrevistado
    "S_002",       # Edad
    "S_101",       # Nivel educativo (recodificado/armonizado)
    "S_200",       # Estado ocupacional
    "S_301",       # Nivel socioeconómico (apreciación encuestador)
    "S_700",       # Religión
    "S_701",       # Práctica religiosa

    # --- Identificación y ponderación ---
    "X_001",       # País (código numérico idenpa)
    "X_002",       # Año de investigación
    "X_004",       # Región / área geográfica
    "X_008",       # Tamaño de ciudad
    "X_020",       # Factor de ponderación muestral
]

print(f"Total de variables seleccionadas: {len(VARIABLES_SELECCIONADAS)}")
print("  (1 target + 39 features/identificadores/ponderación)")

Total de variables seleccionadas: 40
  (1 target + 39 features/identificadores/ponderación)


In [43]:
# =============================================================================
# CELDA 3 — Lectura del CSV y construcción del mapeo
# =============================================================================

def construir_mapeo_variables(ruta_csv: Path, variables_objetivo: list) -> dict:
    """
    Lee latinobarometro_variables.csv (formato largo, sep=';') y construye:

        {
            "A_003_031": {
                "titulo_es": "Satisfacción con la democracia",
                "olas": {
                    "LAT1995": "p21",
                    "LAT1996": "p20",
                    ...
                }
            },
            ...
        }

    Columnas del CSV: year | variable | question | label
    - variable : código estandarizado (ej. A_003_031)
    - question : nombre de la columna en el .dta de ese año (ej. p21)
    - year     : año numérico (ej. 1995) → se convierte a clave LAT1995
    - label    : descripción en español
    """
    df = pd.read_csv(ruta_csv, sep=";", encoding="utf-8", dtype=str, keep_default_na=False)

    variables_obj_set = set(variables_objetivo)
    df_filtrado = df[df["variable"].str.strip().isin(variables_obj_set)].copy()

    mapeo = {}
    for codigo_var, grupo in df_filtrado.groupby("variable"):
        olas = {}
        titulo_es = ""
        for _, fila in grupo.iterrows():
            year_str = str(fila["year"]).strip()
            question = str(fila["question"]).strip()
            if year_str and question:
                olas[f"LAT{year_str}"] = question
            if not titulo_es:
                titulo_es = str(fila["label"]).strip()

        mapeo[codigo_var.strip()] = {
            "titulo_es": titulo_es,
            "olas": olas,
        }

    no_encontradas = variables_obj_set - set(mapeo.keys())
    if no_encontradas:
        print(f"⚠️  Variables no encontradas en el CSV: {sorted(no_encontradas)}")

    return mapeo


# Construir el mapeo
print("Construyendo diccionario de mapeo desde el CSV de variables...")
MAPEO_VARIABLES = construir_mapeo_variables(ARCHIVO_VARIABLES_LB, VARIABLES_SELECCIONADAS)

print(f"✓ Mapeo construido para {len(MAPEO_VARIABLES)} variables.")

#print(MAPEO_VARIABLES)

# Inspección de ejemplo
var_ejemplo = "A_003_031"
info_ejemplo = MAPEO_VARIABLES.get(var_ejemplo)
if info_ejemplo:
    print(f"\nEjemplo — {var_ejemplo}:")
    print(f"  Título ES: {info_ejemplo['titulo_es']}")
    print("  Nombres originales por ola (question):")
    for ola, nombre in list(info_ejemplo["olas"].items())[:6]:
        print(f"    {ola}: {nombre}")
    print("    ...")
else:
    print(f"\n⚠️  '{var_ejemplo}' no encontrada en el CSV.")

Construyendo diccionario de mapeo desde el CSV de variables...
✓ Mapeo construido para 40 variables.

Ejemplo — A_003_031:
  Título ES: Satisfacción con la democracia
  Nombres originales por ola (question):
    LAT1995: p21
    LAT1996: p20
    LAT1997: sp32
    LAT1998: sp29
    LAT2000: P30ST
    LAT2001: p45st
    ...


## 3. Carga y consolidación de archivos Latinobarómetro

Esta sección recorre todos los archivos `.dta` de la carpeta `data/latinobarometro/`,
extrae las columnas correspondientes a las variables seleccionadas para cada ola,
renombra las columnas con los códigos estandarizados y consolida todo en un único
DataFrame. Cada archivo se libera de memoria (`gc.collect()`) tras su procesamiento
para minimizar el consumo de RAM.

In [44]:
# =============================================================================
# CELDA 4 — Función auxiliar: inferir la clave de ola desde el nombre del archivo
# =============================================================================

def inferir_clave_ola(nombre_archivo: str) -> str | None:
    """
    Extrae la clave de ola (ej. 'LAT2022') a partir del nombre de un archivo .dta.
    """
    match = re.search(r"(1[9][0-9]{2}|20[0-9]{2})", nombre_archivo)
    return f"LAT{match.group(1)}" if match else None


# Prueba de la función auxiliar
ejemplos = [
    "Latinobarometro_2022.dta",
    "latinobarometro2005_v2.dta",
    "LB_1997.dta",
    "Latinobarometro2020Oct.dta",
    "datos_sin_año.dta",
]
print("Prueba de inferencia de claves de ola:")
for ej in ejemplos:
    print(f"  {ej:45s} → {inferir_clave_ola(ej)}")

Prueba de inferencia de claves de ola:
  Latinobarometro_2022.dta                      → LAT2022
  latinobarometro2005_v2.dta                    → LAT2005
  LB_1997.dta                                   → LAT1997
  Latinobarometro2020Oct.dta                    → LAT2020
  datos_sin_año.dta                             → None


In [45]:
# =============================================================================
# CELDA 5 — Descubrimiento y validación de archivos .dta
# =============================================================================

# Obtener la lista de archivos .dta disponibles en la carpeta
archivos_dta = sorted(CARPETA_LB.glob("*.dta"))

if not archivos_dta:
    raise FileNotFoundError(
        f"No se encontraron archivos .dta en {CARPETA_LB}. "
        "Verificar que los archivos estén en la carpeta correcta."
    )

print(f"Archivos .dta encontrados: {len(archivos_dta)}")
print()

# Verificar que cada archivo tiene una clave de ola reconocida
archivos_validos = []
archivos_sin_clave = []

for archivo in archivos_dta:
    clave = inferir_clave_ola(archivo.name)
    if clave is None:
        archivos_sin_clave.append(archivo.name)
    else:
        archivos_validos.append((archivo, clave))

# Mostrar tabla de archivos válidos
print(f"{'Archivo':<55} {'Clave de ola':<12} {'En mapeo Excel'}")
print("-" * 82)
for archivo, clave in archivos_validos:
    en_mapeo = "✓" if any(
        clave in info["olas"] for info in MAPEO_VARIABLES.values()
    ) else "⚠️ no en mapeo"
    print(f"  {archivo.name:<53} {clave:<12} {en_mapeo}")

if archivos_sin_clave:
    print(f"\n⚠️  Archivos sin año detectado (serán omitidos): {archivos_sin_clave}")

print(f"\n✓ {len(archivos_validos)} archivos listos para procesar.")

Archivos .dta encontrados: 24

Archivo                                                 Clave de ola En mapeo Excel
----------------------------------------------------------------------------------
  Latinobarometro2013.dta                               LAT2013      ✓
  Latinobarometro2016Esp_v20170205.dta                  LAT2016      ✓
  Latinobarometro2017Esp_v20180117.dta                  LAT2017      ✓
  Latinobarometro_1995_datos_esp_v2014_06_27.dta        LAT1995      ✓
  Latinobarometro_1996_datos_esp_v2014_06_27.dta        LAT1996      ✓
  Latinobarometro_1997_datos_esp_v2014_06_27.dta        LAT1997      ✓
  Latinobarometro_1998_datos_esp_v2014_06_27.dta        LAT1998      ✓
  Latinobarometro_2000_datos_esp_v2014_06_27.dta        LAT2000      ✓
  Latinobarometro_2001_datos_esp_v2014_06_27.dta        LAT2001      ✓
  Latinobarometro_2002_datos_esp_v2014_06_27.dta        LAT2002      ✓
  Latinobarometro_2003_datos_esp_v2014_06_27.dta        LAT2003      ✓
  Latinobarometro_200

In [46]:
# =============================================================================
# CELDA 6 — Función principal: procesar un archivo .dta y extraer variables
# =============================================================================

def _detectar_encoding(ruta_archivo: Path) -> dict:
    """
    Prueba encodings en orden hasta encontrar uno que lea los metadatos sin error.
    Devuelve el kwarg a pasar a read_dta (ej. {} o {"encoding": "latin1"}).
    """
    for enc in (None, "latin1"):
        kwargs = {} if enc is None else {"encoding": enc}
        try:
            pyreadstat.read_dta(str(ruta_archivo), metadataonly=True, **kwargs)
            return kwargs
        except Exception:
            continue
    raise IOError(f"No se pudo determinar el encoding de {ruta_archivo.name}")


def procesar_ola_dta(
    ruta_archivo: Path,
    clave_ola: str,
    mapeo_variables: dict,
) -> pd.DataFrame:
    """
    Lee solo las columnas necesarias de un archivo .dta de Latinobarómetro:

    1. Detecta el encoding leyendo solo metadatos (sin cargar datos).
    2. Resuelve los nombres de columna de forma case-insensitive.
    3. Lee únicamente las columnas requeridas con usecols.

    Las variables sin correspondencia en esta ola se agregan como NaN.
    """
    # -------------------------------------------------------------------------
    # Paso 1: Determinar qué columnas (question) corresponden a esta ola
    # -------------------------------------------------------------------------
    rename_map = {}       # nombre_en_dta (original del CSV) → código estandarizado
    variables_ausentes = []

    for codigo_std, info in mapeo_variables.items():
        if clave_ola in info["olas"]:
            rename_map[info["olas"][clave_ola]] = codigo_std
        else:
            variables_ausentes.append(codigo_std)

    # -------------------------------------------------------------------------
    # Paso 2: Leer metadatos para obtener nombres reales de columna (sin datos)
    # -------------------------------------------------------------------------
    enc_kwargs = _detectar_encoding(ruta_archivo)
    _, meta = pyreadstat.read_dta(str(ruta_archivo), metadataonly=True, **enc_kwargs)

    # Lista de columnas
    #columnas_dta = meta.column_names

    # Guardar en CSV
    #pd.DataFrame({"columna": columnas_dta}).to_csv(
    #    "..\data\\columnas_dta.csv",
    #    index=False,
    #    encoding="utf-8-sig"
    #)


    # -------------------------------------------------------------------------
    # Paso 3: Resolver nombres reales (case-insensitive)
    # -------------------------------------------------------------------------
    cols_lower_a_real = {col.lower(): col for col in meta.column_names}

    rename_map_resuelto = {}   # nombre_real_en_dta → código estandarizado
    variables_no_en_dta = []

    for nombre_csv, codigo_std in rename_map.items():
        nombre_real = cols_lower_a_real.get(nombre_csv.lower())
        if nombre_real:
            rename_map_resuelto[nombre_real] = codigo_std
        else:
            variables_no_en_dta.append((nombre_csv, codigo_std))
            variables_ausentes.append(codigo_std)

    # -------------------------------------------------------------------------
    # Paso 4: Leer SOLO las columnas necesarias
    # -------------------------------------------------------------------------
    columnas_a_leer = list(rename_map_resuelto.keys())
    df_raw, _ = pyreadstat.read_dta(
        str(ruta_archivo),
        usecols=columnas_a_leer,
        **enc_kwargs,
    )

    # -------------------------------------------------------------------------
    # Paso 5: Renombrar, agregar ausentes y ordenar
    # -------------------------------------------------------------------------
    df_ola = df_raw.rename(columns=rename_map_resuelto)
    del df_raw
    gc.collect()

    for codigo_std in variables_ausentes:
        if codigo_std not in df_ola.columns:
            df_ola[codigo_std] = np.nan

    df_ola.insert(0, "ola", clave_ola)
    cols_ord = ["ola"] + sorted(c for c in df_ola.columns if c != "ola")
    df_ola = df_ola[cols_ord]

    if variables_no_en_dta:
        print(f"Total de variables no encontradas en el .dta: {len(variables_no_en_dta)}")
        for nombre_orig, codigo_std in variables_no_en_dta:
            print(f"    ⚠️  [{clave_ola}] '{nombre_orig}' (→{codigo_std}) no encontrada en el .dta.")

    return df_ola


print("✓ Función 'procesar_ola_dta' definida correctamente.")

✓ Función 'procesar_ola_dta' definida correctamente.


In [47]:
# archivos_validos

In [48]:
# =============================================================================
# CELDA 7 — Prueba de procesamiento con el primer archivo (verificación previa)
# =============================================================================
# Se prueba el procesamiento con el primer archivo antes de ejecutar el ciclo
# completo, para detectar problemas de formato o codificación de forma temprana.

archivo_prueba, clave_prueba = archivos_validos[22]
print(f"Probando con: {archivo_prueba.name} ({clave_prueba})")
print("-" * 60)

df_prueba = procesar_ola_dta(archivo_prueba, clave_prueba, MAPEO_VARIABLES)

print(f"\n  Filas     : {len(df_prueba):,}")
print(f"  Columnas  : {len(df_prueba.columns)}")
print(f"  Columnas con todos NaN en esta ola:")
cols_nan = [c for c in df_prueba.columns if df_prueba[c].isna().all()]
for c in cols_nan:
    print(f"    - {c} (variable ausente en {clave_prueba})")

print("\nPrimeras 3 filas de la prueba:")
df_prueba.head(3)

Probando con: Latinobarometro_2023_Esp_Stata_v1_0.dta (LAT2023)
------------------------------------------------------------

  Filas     : 19,205
  Columnas  : 41
  Columnas con todos NaN en esta ola:
    - A_003_021 (variable ausente en LAT2023)
    - D_001_061 (variable ausente en LAT2023)
    - D_001_131 (variable ausente en LAT2023)
    - G_002_011 (variable ausente en LAT2023)
    - H_002_131 (variable ausente en LAT2023)

Primeras 3 filas de la prueba:


,ola,A_001_001,A_003_021,A_003_031,A_007_001,A_007_071,B_001_101,B_006_061,C_001_031,C_003_003_011,...,S_101,S_200,S_301,S_700,S_701,X_001,X_002,X_004,X_008,X_020
0,LAT2023,1,NaN,4,4,6,1,2,26,1,...,5,5,3,1,4,32,23,32001,8,0.881633
1,LAT2023,2,NaN,4,1,7,1,2,5,1,...,4,1,3,1,2,32,23,32001,8,0.881633
2,LAT2023,1,NaN,3,1,5,1,2,10,1,...,4,1,3,1,3,32,23,32001,8,0.881633


In [49]:
# =============================================================================
# CELDA 8 — Ciclo principal: procesamiento de todas las olas
# =============================================================================
# Estrategia de memoria:
#   1. Procesar un archivo .dta a la vez.
#   2. Al terminar cada archivo, el DataFrame completo se descarta (del df_raw)
#      dentro de `procesar_ola_dta` y gc.collect() libera la memoria.
#   3. Solo el DataFrame reducido (variables seleccionadas) se acumula en `bloques`.
#   4. Al final, pd.concat une todos los bloques en un único DataFrame consolidado.

bloques = []  # Lista de DataFrames, uno por ola
log_procesamiento = []  # Registro para el reporte final

print("Iniciando procesamiento de olas de Latinobarómetro...")
print("=" * 70)

for archivo, clave_ola in tqdm(archivos_validos, desc="Procesando olas"):
    try:
        print(f"\nProcesando {clave_ola} ({archivo.name})...")
        df_ola = procesar_ola_dta(archivo, clave_ola, MAPEO_VARIABLES)

        n_filas = len(df_ola)
        n_vars_con_datos = sum(
            1 for c in df_ola.columns
            if c not in ("ola",) and not df_ola[c].isna().all()
        )
        n_vars_ausentes = sum(
            1 for c in df_ola.columns
            if c not in ("ola",) and df_ola[c].isna().all()
        )

        bloques.append(df_ola)
        log_procesamiento.append({
            "ola"         : clave_ola,
            "archivo"     : archivo.name,
            "n_registros" : n_filas,
            "vars_con_datos": n_vars_con_datos,
            "vars_ausentes" : n_vars_ausentes,
            "estado"      : "OK",
        })

        # Liberar el DataFrame de la ola de la variable local
        # (el objeto aún vive en `bloques`, pero liberamos la referencia local)
        del df_ola
        gc.collect()

    except Exception as e:
        print(f"\n  ✗ Error en {clave_ola} ({archivo.name}): {e}")
        log_procesamiento.append({
            "ola"         : clave_ola,
            "archivo"     : archivo.name,
            "n_registros" : 0,
            "vars_con_datos": 0,
            "vars_ausentes" : 0,
            "estado"      : f"ERROR: {e}",
        })

print("\n✓ Procesamiento de archivos completado.")
print(f"  Olas procesadas exitosamente: {sum(1 for r in log_procesamiento if r['estado'] == 'OK')}")
print(f"  Olas con error              : {sum(1 for r in log_procesamiento if 'ERROR' in r['estado'])}")

Iniciando procesamiento de olas de Latinobarómetro...


Procesando olas:   0%|          | 0/24 [00:00<?, ?it/s]


Procesando LAT2013 (Latinobarometro2013.dta)...


Procesando olas:   4%|▍         | 1/24 [00:00<00:17,  1.30it/s]


Procesando LAT2016 (Latinobarometro2016Esp_v20170205.dta)...


...


Procesando LAT2023 (Latinobarometro_2023_Esp_Stata_v1_0.dta)...


Procesando olas:  96%|█████████▌| 23/24 [00:10<00:00,  2.18it/s]


Procesando LAT2024 (Latinobarometro_2024_Stata_esp_v20250817.dta)...


Procesando olas: 100%|██████████| 24/24 [00:11<00:00,  2.09it/s]


✓ Procesamiento de archivos completado.
  Olas procesadas exitosamente: 24
  Olas con error              : 0


In [50]:
# =============================================================================
# CELDA 9 — Reporte de procesamiento por ola
# =============================================================================

df_log = pd.DataFrame(log_procesamiento)
print("Reporte de procesamiento por ola:")
print("=" * 85)
print(f"  {'Ola':<10} {'Registros':>10} {'Vars con datos':>15} {'Vars ausentes':>14} {'Estado'}")
print("-" * 85)
for _, fila in df_log.iterrows():
    icono = "✓" if fila["estado"] == "OK" else "✗"
    print(
        f"  {icono} {fila['ola']:<9} "
        f"{fila['n_registros']:>10,} "
        f"{fila['vars_con_datos']:>15} "
        f"{fila['vars_ausentes']:>14} "
        f"{fila['estado']}"
    )

print("-" * 85)
print(f"  {'TOTAL':<10} {df_log['n_registros'].sum():>10,}")

Reporte de procesamiento por ola:
  Ola         Registros  Vars con datos  Vars ausentes Estado
-------------------------------------------------------------------------------------
  ✓ LAT2013       22,663              39              1 OK
  ✓ LAT2016       20,204              35              5 OK
  ✓ LAT2017       20,200              35              5 OK
  ✓ LAT1995        9,069              28             12 OK
  ✓ LAT1996       21,200              29             11 OK
  ✓ LAT1997       20,243              29             11 OK
  ✓ LAT1998       20,331              28             12 OK
  ✓ LAT2000       18,038              27             13 OK
  ✓ LAT2001       20,631              34              6 OK
  ✓ LAT2002       21,006              35              5 OK
  ✓ LAT2003       21,133              35              5 OK
  ✓ LAT2004       22,096              38              2 OK
  ✓ LAT2005       20,222              39              1 OK
  ✓ LAT2006       22,708              38           

In [51]:
# =============================================================================
# CELDA 10 — Consolidación de todos los bloques en un único DataFrame
# =============================================================================

print("Consolidando todos los bloques en un único DataFrame...")

df_lb = pd.concat(bloques, axis=0, ignore_index=True, sort=False)

# Liberar la lista de bloques (ya no es necesaria)
del bloques
gc.collect()

print(f"✓ DataFrame consolidado:")
print(f"  Dimensiones  : {df_lb.shape[0]:,} filas × {df_lb.shape[1]} columnas")
print(f"  Memoria aprox: {df_lb.memory_usage(deep=True).sum() / 1_048_576:.1f} MB")
print(f"  Columnas     : {list(df_lb.columns)}")

Consolidando todos los bloques en un único DataFrame...
✓ DataFrame consolidado:
  Dimensiones  : 489,771 filas × 41 columnas
  Memoria aprox: 522.3 MB
  Columnas     : ['ola', 'A_001_001', 'A_003_021', 'A_003_031', 'A_007_001', 'A_007_071', 'B_001_101', 'B_006_061', 'C_001_031', 'C_003_003_011', 'C_006_003_011', 'D_001_001', 'D_001_021', 'D_001_041', 'D_001_061', 'D_001_091', 'D_001_131', 'G_002_011', 'G_005_001', 'H_001_011', 'H_002_011', 'H_002_031', 'H_002_041', 'H_002_101', 'H_002_111', 'H_002_131', 'H_002_161', 'H_002_241', 'I_001_001', 'S_001', 'S_002', 'S_101', 'S_200', 'S_301', 'S_700', 'S_701', 'X_001', 'X_002', 'X_004', 'X_008', 'X_020']


## 4. Extracción del listado de países del estudio

A partir del DataFrame consolidado de Latinobarómetro se obtiene el listado
de países únicos. Este listado se utilizará posteriormente para filtrar
los indicadores de V-Dem a los países relevantes para el estudio.

In [52]:
# =============================================================================
# CELDA 11 — Extracción del listado de países y mapeo a ISO3
# =============================================================================

# Mapeo de códigos numéricos de Latinobarómetro (idenpa) a nombres e ISO3
# Fuente: documentación oficial de Latinobarómetro
MAPEO_PAISES_LB = {
    32  : {"nombre": "Argentina",           "iso3": "ARG"},
    68  : {"nombre": "Bolivia",              "iso3": "BOL"},
    76  : {"nombre": "Brasil",               "iso3": "BRA"},
    152 : {"nombre": "Chile",                "iso3": "CHL"},
    170 : {"nombre": "Colombia",             "iso3": "COL"},
    188 : {"nombre": "Costa Rica",           "iso3": "CRI"},
    214 : {"nombre": "República Dominicana", "iso3": "DOM"},
    218 : {"nombre": "Ecuador",              "iso3": "ECU"},
    222 : {"nombre": "El Salvador",          "iso3": "SLV"},
    320 : {"nombre": "Guatemala",            "iso3": "GTM"},
    340 : {"nombre": "Honduras",             "iso3": "HND"},
    484 : {"nombre": "México",               "iso3": "MEX"},
    558 : {"nombre": "Nicaragua",            "iso3": "NIC"},
    591 : {"nombre": "Panamá",               "iso3": "PAN"},
    600 : {"nombre": "Paraguay",             "iso3": "PRY"},
    604 : {"nombre": "Perú",                 "iso3": "PER"},
    858 : {"nombre": "Uruguay",              "iso3": "URY"},
    862 : {"nombre": "Venezuela",            "iso3": "VEN"},
}

# Obtener códigos numéricos únicos presentes en los datos
# X_001 contiene el código numérico del país (idenpa)
codigos_en_datos = (
    df_lb["X_001"]
    .dropna()
    .astype(float)
    .astype(int)
    .unique()
)
codigos_en_datos = sorted(codigos_en_datos)

# Construir el DataFrame de países del estudio
registros_paises = []
for codigo in codigos_en_datos:
    if codigo in MAPEO_PAISES_LB:
        info = MAPEO_PAISES_LB[codigo]
        registros_paises.append({
            "idenpa": codigo,
            "nombre": info["nombre"],
            "iso3"  : info["iso3"],
        })
    else:
        print(f"⚠️  Código de país no reconocido en el mapeo: {codigo}")
        registros_paises.append({
            "idenpa": codigo,
            "nombre": f"Desconocido_{codigo}",
            "iso3"  : "UNK",
        })

df_paises = pd.DataFrame(registros_paises)
LISTA_ISO3_ESTUDIO = df_paises["iso3"].tolist()

print(f"Países identificados en los datos de Latinobarómetro: {len(df_paises)}")
print()
print(df_paises.to_string(index=False))
print(f"\nCódigos ISO3 para filtrado V-Dem:")
print(f"  {LISTA_ISO3_ESTUDIO}")

⚠️  Código de país no reconocido en el mapeo: 724
Países identificados en los datos de Latinobarómetro: 19

 idenpa               nombre iso3
     32            Argentina  ARG
     68              Bolivia  BOL
     76               Brasil  BRA
    152                Chile  CHL
    170             Colombia  COL
    188           Costa Rica  CRI
    214 República Dominicana  DOM
    218              Ecuador  ECU
    222          El Salvador  SLV
    320            Guatemala  GTM
    340             Honduras  HND
    484               México  MEX
    558            Nicaragua  NIC
    591               Panamá  PAN
    600             Paraguay  PRY
    604                 Perú  PER
    724      Desconocido_724  UNK
    858              Uruguay  URY
    862            Venezuela  VEN

Códigos ISO3 para filtrado V-Dem:
  ['ARG', 'BOL', 'BRA', 'CHL', 'COL', 'CRI', 'DOM', 'ECU', 'SLV', 'GTM', 'HND', 'MEX', 'NIC', 'PAN', 'PRY', 'PER', 'UNK', 'URY', 'VEN']


In [53]:
# =============================================================================
# CELDA 12 — Agregar columnas de nombre de país e ISO3 al DataFrame consolidado
# =============================================================================

# Crear mapeo: idenpa (float) → nombre e iso3
mapa_idenpa_a_nombre = {float(k): v["nombre"] for k, v in MAPEO_PAISES_LB.items()}
mapa_idenpa_a_iso3   = {float(k): v["iso3"]   for k, v in MAPEO_PAISES_LB.items()}

df_lb["pais_nombre"] = df_lb["X_001"].map(mapa_idenpa_a_nombre)
df_lb["pais_iso3"]   = df_lb["X_001"].map(mapa_idenpa_a_iso3)

# Verificar cobertura del mapeo
n_sin_nombre = df_lb["pais_nombre"].isna().sum()
n_sin_iso3   = df_lb["pais_iso3"].isna().sum()

print(f"Columnas 'pais_nombre' e 'pais_iso3' agregadas al DataFrame.")
print(f"  Registros sin nombre mapeado : {n_sin_nombre:,}")
print(f"  Registros sin ISO3 mapeado   : {n_sin_iso3:,}")

if n_sin_nombre > 0:
    codigos_no_mapeados = (
        df_lb[df_lb["pais_nombre"].isna()]["X_001"]
        .dropna().unique()
    )
    print(f"  ⚠️  Códigos sin mapear: {codigos_no_mapeados}")

Columnas 'pais_nombre' e 'pais_iso3' agregadas al DataFrame.
  Registros sin nombre mapeado : 32,272
  Registros sin ISO3 mapeado   : 32,272
  ⚠️  Códigos sin mapear: [724]


## 5. Exportación del DataFrame de Latinobarómetro

In [54]:
# =============================================================================
# CELDA 13 — Exportación del DataFrame consolidado a CSV
# =============================================================================

print(f"Exportando DataFrame de Latinobarómetro a: {SALIDA_LB}")

# Crear la carpeta de destino si no existe
SALIDA_LB.parent.mkdir(parents=True, exist_ok=True)

df_lb.to_csv(
    SALIDA_LB,
    index=False,
    encoding="utf-8-sig",  # UTF-8 con BOM para compatibilidad con Excel
)

# Verificación del archivo generado
tamanio_mb = SALIDA_LB.stat().st_size / 1_048_576

print(f"✓ Archivo generado exitosamente.")
print(f"  Ruta    : {SALIDA_LB}")
print(f"  Tamaño  : {tamanio_mb:.1f} MB")
print(f"  Filas   : {len(df_lb):,}")
print(f"  Columnas: {len(df_lb.columns)}")

# Verificación de integridad: releer el CSV y comparar dimensiones
df_verificacion = pd.read_csv(SALIDA_LB, nrows=5)
print(f"\nVerificación de lectura (primeras 5 filas):")
print(f"  Columnas leídas: {len(df_verificacion.columns)} ← debe coincidir con {len(df_lb.columns)}")
del df_verificacion

Exportando DataFrame de Latinobarómetro a: ..\data\latinobarometro.csv
✓ Archivo generado exitosamente.
  Ruta    : ..\data\latinobarometro.csv
  Tamaño  : 57.6 MB
  Filas   : 489,771
  Columnas: 43

Verificación de lectura (primeras 5 filas):
  Columnas leídas: 43 ← debe coincidir con 43


## 6. Carga y filtrado de V-Dem

Se lee el archivo `V-Dem-CY-Core-v16.csv`, se filtran los registros
correspondientes a los países del estudio y al período 1995–2024,
y se seleccionan las columnas definidas en la metodología.

**Nota sobre versiones de variables:** Se utilizan exclusivamente las columnas
base sin sufijo (ej. `v2x_polyarchy`, NO `v2x_polyarchy_osp`), que corresponden
a las estimaciones del modelo de medición en escala de intervalo continua,
tal como se especifica en la metodología de investigación.

**Nota sobre dirección de índices:** Algunos índices tienen dirección invertida
(mayor valor = peor situación democrática). Estas variables están documentadas
en `variables_seleccionadas_metodologia.md` y se señalan en el reporte de
esta sección.

In [55]:
# =============================================================================
# CELDA 14 — Definición de columnas V-Dem a seleccionar
# =============================================================================

# Columnas de identificación (siempre incluir)
COLS_ID_VDEM = [
    "country_name",       # Nombre del país en inglés
    "country_text_id",    # Código ISO3 del país
    "country_id",         # ID numérico V-Dem
    "year",               # Año
    "COWcode",            # Correlates of War code (útil para merge con otras fuentes)
]

# Índices de alto nivel — SOLO para Regresión Logística Ordinal (baseline)
COLS_HIGHLEVEL_VDEM = [
    "v2x_polyarchy",      # Democracia electoral (poliarquía)
    "v2x_libdem",         # Democracia liberal
    "v2x_partipdem",      # Democracia participativa
    "v2x_delibdem",       # Democracia deliberativa
    "v2x_egaldem",        # Democracia igualitaria
]

# Índices mid-level y especializados — Para XGBoost, CatBoost, LightGBM, TabNet
COLS_MIDLEVEL_VDEM = [
    # Componentes liberales (subcomponentes de v2x_libdem)
    "v2xcl_rol",          # Igualdad ante la ley y libertades individuales
    "v2x_jucon",          # Restricciones judiciales al ejecutivo
    "v2xlg_legcon",       # Restricciones legislativas al ejecutivo

    # Libertad de expresión (subcomponente de v2x_polyarchy)
    "v2x_freexp_altinf",  # Libertad de expresión e información alternativa

    # Corrupción — ⚠️ DIRECCIÓN INVERTIDA: mayor valor = MÁS corrupción
    "v2x_corr",           # Corrupción política (índice global)
    "v2x_execorr",        # Corrupción ejecutiva
    "v2x_pubcorr",        # Corrupción sector público

    # Neopatrimonialismo — ⚠️ DIRECCIÓN INVERTIDA: mayor valor = PEOR
    "v2x_neopat",         # Neopatrimonialismo
    "v2xnp_regcorr",      # Corrupción del régimen

    # Accountability e instituciones
    "v2x_accountability_osp",  # Accountability (versión OSP [0,1]; la base es z-score)
    "v2xcs_ccsi",         # Índice de sociedad civil
    "v2x_rule",           # Estado de derecho
    "v2x_cspart",         # Participación de la sociedad civil

    # Igualdad y exclusión
    "v2x_egal",           # Componente igualitario
    "v2xeg_eqdr",         # Distribución igualitaria de recursos

    # Exclusión — ⚠️ DIRECCIÓN INVERTIDA: mayor valor = MÁS exclusión
    "v2xpe_exlsocgr",     # Exclusión por grupo social
    "v2xpe_exlecon",      # Exclusión por posición económica

    # Género
    "v2x_gender",         # Empoderamiento político de mujeres
]

# Lista completa de columnas a retener
COLS_VDEM_TOTAL = COLS_ID_VDEM + COLS_HIGHLEVEL_VDEM + COLS_MIDLEVEL_VDEM

# Índices con dirección invertida (documentado para el agente de XAI)
COLS_INVERTIDAS_VDEM = [
    "v2x_corr", "v2x_execorr", "v2x_pubcorr",
    "v2x_neopat", "v2xnp_regcorr",
    "v2xpe_exlsocgr", "v2xpe_exlecon",
]

print(f"Columnas V-Dem a seleccionar: {len(COLS_VDEM_TOTAL)}")
print(f"  Identificación     : {len(COLS_ID_VDEM)}")
print(f"  High-level indices : {len(COLS_HIGHLEVEL_VDEM)}")
print(f"  Mid-level indices  : {len(COLS_MIDLEVEL_VDEM)}")
print(f"\nÍndices con dirección invertida (⚠️): {COLS_INVERTIDAS_VDEM}")

Columnas V-Dem a seleccionar: 28
  Identificación     : 5
  High-level indices : 5
  Mid-level indices  : 18

Índices con dirección invertida (⚠️): ['v2x_corr', 'v2x_execorr', 'v2x_pubcorr', 'v2x_neopat', 'v2xnp_regcorr', 'v2xpe_exlsocgr', 'v2xpe_exlecon']


In [56]:
# =============================================================================
# CELDA 15 — Lectura del CSV de V-Dem
# =============================================================================

print(f"Leyendo V-Dem CY-Core desde: {ARCHIVO_VDEM}")
print("(Se leen solo las columnas seleccionadas para optimizar memoria)")
print()

# Leer solo los encabezados (nrows=0) para verificar qué columnas existen
# Pandas maneja correctamente las comillas en el CSV
cols_disponibles_vdem = set(pd.read_csv(ARCHIVO_VDEM, nrows=0, encoding="utf-8").columns)

cols_a_leer       = [c for c in COLS_VDEM_TOTAL if c in cols_disponibles_vdem]
cols_ausentes_vdem = [c for c in COLS_VDEM_TOTAL if c not in cols_disponibles_vdem]

print(f"Columnas solicitadas: {len(COLS_VDEM_TOTAL)}")
print(f"Columnas encontradas: {len(cols_a_leer)}")
print(f"Columnas no encontradas: {len(cols_ausentes_vdem)}")
if cols_ausentes_vdem:
    print(f"⚠️  Columnas seleccionadas NO encontradas en el CSV V-Dem:")
    for c in cols_ausentes_vdem:
        print(f"    - {c}")
    print()

# Leer solo las columnas necesarias
df_vdem_completo = pd.read_csv(
    ARCHIVO_VDEM,
    usecols=cols_a_leer,
    low_memory=False,
    encoding="utf-8",
)

print(f"✓ V-Dem leído correctamente.")
print(f"  Columnas leídas  : {df_vdem_completo.shape[1]} de {len(COLS_VDEM_TOTAL)} solicitadas")
print(f"  Dimensiones      : {df_vdem_completo.shape[0]:,} filas × {df_vdem_completo.shape[1]} columnas")
print(f"  Rango de años    : {df_vdem_completo['year'].min()} – {df_vdem_completo['year'].max()}")
print(f"  Países totales   : {df_vdem_completo['country_name'].nunique()}")

Leyendo V-Dem CY-Core desde: ..\data\v-dem\V-Dem-CY-Core-v16.csv
(Se leen solo las columnas seleccionadas para optimizar memoria)

Columnas solicitadas: 28
Columnas encontradas: 28
Columnas no encontradas: 0
✓ V-Dem leído correctamente.
  Columnas leídas  : 28 de 28 solicitadas
  Dimensiones      : 28,092 filas × 28 columnas
  Rango de años    : 1789 – 2025
  Países totales   : 202


In [57]:
# =============================================================================
# CELDA 16 — Filtrado por países del estudio y período 1995–2024
# =============================================================================

AÑO_INICIO = 1995
AÑO_FIN    = 2024

# Verificar qué ISO3 del estudio están en V-Dem
iso3_en_vdem     = set(df_vdem_completo["country_text_id"].unique())
iso3_encontrados = [iso3 for iso3 in LISTA_ISO3_ESTUDIO if iso3 in iso3_en_vdem]
iso3_no_en_vdem  = [iso3 for iso3 in LISTA_ISO3_ESTUDIO if iso3 not in iso3_en_vdem]

if iso3_no_en_vdem:
    print(f"⚠️  Países del estudio no encontrados en V-Dem: {iso3_no_en_vdem}")

# Aplicar filtro
mascara_paises = df_vdem_completo["country_text_id"].isin(iso3_encontrados)
mascara_anios  = (df_vdem_completo["year"] >= AÑO_INICIO) & (df_vdem_completo["year"] <= AÑO_FIN)

df_vdem = df_vdem_completo[mascara_paises & mascara_anios].copy()

# Liberar el DataFrame completo
del df_vdem_completo
gc.collect()

# Ordenar por país y año
df_vdem = df_vdem.sort_values(["country_text_id", "year"]).reset_index(drop=True)

print(f"Filtro aplicado: países del estudio ({len(iso3_encontrados)}) + años {AÑO_INICIO}–{AÑO_FIN}")
print(f"\n✓ DataFrame V-Dem filtrado:")
print(f"  Dimensiones  : {df_vdem.shape[0]:,} filas × {df_vdem.shape[1]} columnas")
print(f"  Países       : {df_vdem['country_name'].nunique()}")
print(f"  Rango de años: {df_vdem['year'].min()} – {df_vdem['year'].max()}")

print(f"\nCobertura por país (filas = años disponibles):")
cobertura = (
    df_vdem.groupby(["country_text_id", "country_name"])
    .size()
    .reset_index(name="n_años")
    .sort_values("country_text_id")
)
print(cobertura.to_string(index=False))

⚠️  Países del estudio no encontrados en V-Dem: ['UNK']
Filtro aplicado: países del estudio (18) + años 1995–2024

✓ DataFrame V-Dem filtrado:
  Dimensiones  : 540 filas × 28 columnas
  Países       : 18
  Rango de años: 1995 – 2024

Cobertura por país (filas = años disponibles):
country_text_id       country_name  n_años
            ARG          Argentina      30
            BOL            Bolivia      30
            BRA             Brazil      30
            CHL              Chile      30
            COL           Colombia      30
            CRI         Costa Rica      30
            DOM Dominican Republic      30
            ECU            Ecuador      30
            GTM          Guatemala      30
            HND           Honduras      30
            MEX             Mexico      30
            NIC          Nicaragua      30
            PAN             Panama      30
            PER               Peru      30
            PRY           Paraguay      30
            SLV        El Salva

In [58]:
# =============================================================================
# CELDA 17 — Verificación de cobertura de variables V-Dem seleccionadas
# =============================================================================

print("Cobertura de variables V-Dem en el dataset filtrado:")
print(f"  (países AL, {AÑO_INICIO}–{AÑO_FIN}, {len(df_vdem)} registros totales)")
print()
print(f"  {'Variable':<30} {'% completitud':>14} {'Min':>8} {'Max':>8} {'Invertida'}")
print("-" * 80)

for col in COLS_MIDLEVEL_VDEM + COLS_HIGHLEVEL_VDEM:
    if col not in df_vdem.columns:
        print(f"  {col:<30} {'NO DISPONIBLE':>14}")
        continue
    completitud = (1 - df_vdem[col].isna().mean()) * 100
    val_min = df_vdem[col].min() if completitud > 0 else np.nan
    val_max = df_vdem[col].max() if completitud > 0 else np.nan
    es_inv = "⚠️ INVERTIDA" if col in COLS_INVERTIDAS_VDEM else ""
    print(
        f"  {col:<30} {completitud:>13.1f}% "
        f"{val_min:>8.3f} {val_max:>8.3f}  {es_inv}"
    )

Cobertura de variables V-Dem en el dataset filtrado:
  (países AL, 1995–2024, 540 registros totales)

  Variable                        % completitud      Min      Max Invertida
--------------------------------------------------------------------------------
  v2xcl_rol                              100.0%    0.038    0.972  
  v2x_jucon                              100.0%    0.003    0.968  
  v2xlg_legcon                           100.0%    0.024    0.983  
  v2x_freexp_altinf                      100.0%    0.022    0.975  
  v2x_corr                               100.0%    0.041    0.971  ⚠️ INVERTIDA
  v2x_execorr                            100.0%    0.026    0.979  ⚠️ INVERTIDA
  v2x_pubcorr                            100.0%    0.055    0.990  ⚠️ INVERTIDA
  v2x_neopat                             100.0%    0.036    0.989  ⚠️ INVERTIDA
  v2xnp_regcorr                          100.0%    0.028    0.973  ⚠️ INVERTIDA
  v2x_accountability_osp                 100.0%    0.082    0.973  
 

In [59]:
# =============================================================================
# CELDA 18 — Agregar metadatos de dirección de índice como columna auxiliar
# =============================================================================
# Se agrega una columna auxiliar 'vars_invertidas_nota' como referencia
# para el agente de XAI. No se modifica ningún valor numérico.

# Nota: No se invierten los valores en esta etapa.
# La inversión conceptual se documenta aquí para que el pipeline de
# preprocesamiento (siguiente notebook) la aplique de forma controlada.

df_invertidas_meta = pd.DataFrame({
    "variable"   : COLS_INVERTIDAS_VDEM,
    "descripcion": [
        "Corrupción política global — mayor = MÁS corrupción",
        "Corrupción ejecutiva — mayor = MÁS corrupción",
        "Corrupción sector público — mayor = MÁS corrupción",
        "Neopatrimonialismo — mayor = MÁS neopatrimonialismo",
        "Corrupción del régimen — mayor = MÁS corrupción",
        "Exclusión por grupo social — mayor = MÁS exclusión",
        "Exclusión por posición económica — mayor = MÁS exclusión",
    ],
})

print("Registro de variables V-Dem con dirección invertida:")
print("(Documentadas para uso en el pipeline de XAI — los valores NO se modifican aquí)")
print()
print(df_invertidas_meta.to_string(index=False))

Registro de variables V-Dem con dirección invertida:
(Documentadas para uso en el pipeline de XAI — los valores NO se modifican aquí)

      variable                                              descripcion
      v2x_corr      Corrupción política global — mayor = MÁS corrupción
   v2x_execorr            Corrupción ejecutiva — mayor = MÁS corrupción
   v2x_pubcorr       Corrupción sector público — mayor = MÁS corrupción
    v2x_neopat      Neopatrimonialismo — mayor = MÁS neopatrimonialismo
 v2xnp_regcorr          Corrupción del régimen — mayor = MÁS corrupción
v2xpe_exlsocgr       Exclusión por grupo social — mayor = MÁS exclusión
 v2xpe_exlecon Exclusión por posición económica — mayor = MÁS exclusión


## 7. Exportación del DataFrame de V-Dem

In [60]:
# =============================================================================
# CELDA 19 — Exportación del DataFrame de V-Dem a CSV
# =============================================================================

print(f"Exportando DataFrame de V-Dem a: {SALIDA_VDEM}")

SALIDA_VDEM.parent.mkdir(parents=True, exist_ok=True)

df_vdem.to_csv(
    SALIDA_VDEM,
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f",  # 6 decimales para preservar precisión de los índices
)

tamanio_mb_vdem = SALIDA_VDEM.stat().st_size / 1_048_576

print(f"✓ Archivo generado exitosamente.")
print(f"  Ruta    : {SALIDA_VDEM}")
print(f"  Tamaño  : {tamanio_mb_vdem:.1f} MB")
print(f"  Filas   : {len(df_vdem):,}")
print(f"  Columnas: {len(df_vdem.columns)}")

# Verificación de integridad
df_verif_vdem = pd.read_csv(SALIDA_VDEM, nrows=3)
print(f"\nVerificación de lectura (primeras 3 filas):")
print(f"  Columnas leídas: {len(df_verif_vdem.columns)} ← debe coincidir con {len(df_vdem.columns)}")
del df_verif_vdem

Exportando DataFrame de V-Dem a: ..\data\v-dem.csv
✓ Archivo generado exitosamente.
  Ruta    : ..\data\v-dem.csv
  Tamaño  : 0.1 MB
  Filas   : 540
  Columnas: 28

Verificación de lectura (primeras 3 filas):
  Columnas leídas: 28 ← debe coincidir con 28


## 8. Resumen ejecutivo del pipeline de carga

In [61]:
# =============================================================================
# CELDA 20 — Resumen ejecutivo y estadísticas finales
# =============================================================================

print("=" * 70)
print("  RESUMEN EJECUTIVO — PIPELINE DE CARGA DE DATOS")
print("=" * 70)

print("\n📁  LATINOBARÓMETRO")
print(f"    Archivo de salida  : {SALIDA_LB.name}")
print(f"    Total registros    : {len(df_lb):,}")
print(f"    Total columnas     : {len(df_lb.columns)}")
print(f"    Olas incluidas     : {df_lb['ola'].nunique()}")
print(f"    Países incluidos   : {df_lb['pais_iso3'].nunique()}")
print(f"    Rango temporal     : {df_lb['ola'].min()} – {df_lb['ola'].max()}")

# Distribución de registros por ola
print("\n    Registros por ola:")
dist_olas = df_lb.groupby("ola").size().reset_index(name="n")
for _, row in dist_olas.iterrows():
    print(f"      {row['ola']}: {row['n']:,}")

print("\n📁  V-DEM")
print(f"    Archivo de salida  : {SALIDA_VDEM.name}")
print(f"    Total registros    : {len(df_vdem):,}")
print(f"    Total columnas     : {len(df_vdem.columns)}")
print(f"    Países incluidos   : {df_vdem['country_name'].nunique()}")
print(f"    Rango temporal     : {df_vdem['year'].min()} – {df_vdem['year'].max()}")

print("\n✅  Pipeline completado. Archivos listos para el siguiente notebook:")
print(f"    - {SALIDA_LB}")
print(f"    - {SALIDA_VDEM}")
print()
print("📝  Próximo paso: 02_preprocesamiento.ipynb")
print("    - Merge Latinobarómetro × V-Dem por (pais_iso3, año)")
print("    - Inversión de escalas (confianza institucional, índices V-Dem invertidos)")
print("    - Imputación MICE para variables con missingness estructural")
print("    - Partición Time-Based Split (Train: 1995–2018 / Test: 2019–2024)")
print("    - Tratamiento especial de Venezuela (solo train hasta 2017)")
print("=" * 70)

  RESUMEN EJECUTIVO — PIPELINE DE CARGA DE DATOS

📁  LATINOBARÓMETRO
    Archivo de salida  : latinobarometro.csv
    Total registros    : 489,771
    Total columnas     : 43
    Olas incluidas     : 24
    Países incluidos   : 18
    Rango temporal     : LAT1995 – LAT2024

    Registros por ola:
      LAT1995: 9,069
      LAT1996: 21,200
      LAT1997: 20,243
      LAT1998: 20,331
      LAT2000: 18,038
      LAT2001: 20,631
      LAT2002: 21,006
      LAT2003: 21,133
      LAT2004: 22,096
      LAT2005: 20,222
      LAT2006: 22,708
      LAT2007: 22,694
      LAT2008: 22,675
      LAT2009: 22,690
      LAT2010: 22,687
      LAT2011: 20,204
      LAT2013: 22,663
      LAT2015: 20,250
      LAT2016: 20,204
      LAT2017: 20,200
      LAT2018: 20,204
      LAT2020: 20,204
      LAT2023: 19,205
      LAT2024: 19,214

📁  V-DEM
    Archivo de salida  : v-dem.csv
    Total registros    : 540
    Total columnas     : 28
    Países incluidos   : 18
    Rango temporal     : 1995 – 2024

✅  Pipe

In [62]:
# =============================================================================
# CELDA 21 — Registro de versiones de librerías (reproducibilidad)
# =============================================================================

import importlib
import platform
from datetime import datetime

librerias = [
    "pandas", "numpy", "pyreadstat", "openpyxl", "tqdm",
]

print("Registro de entorno de ejecución (reproducibilidad)")
print("-" * 50)
print(f"  Fecha de ejecución : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Python             : {sys.version.split()[0]}")
print(f"  Sistema operativo  : {platform.system()} {platform.release()}")
print()
print("  Versiones de librerías:")
for lib in librerias:
    try:
        mod = importlib.import_module(lib)
        version = getattr(mod, "__version__", "N/D")
        print(f"    {lib:<15}: {version}")
    except ImportError:
        print(f"    {lib:<15}: NO INSTALADA")

Registro de entorno de ejecución (reproducibilidad)
--------------------------------------------------
  Fecha de ejecución : 2026-07-05 19:53:50
  Python             : 3.13.14
  Sistema operativo  : Windows 11

  Versiones de librerías:
    pandas         : 3.0.3
    numpy          : 2.5.0
    pyreadstat     : 1.3.5
    openpyxl       : 3.1.5
    tqdm           : 4.68.3


# Análisis de resultados

In [63]:
df_lb.head(3)

,ola,A_001_001,A_003_021,A_003_031,A_007_001,A_007_071,B_001_101,B_006_061,C_001_031,C_003_003_011,...,S_301,S_700,S_701,X_001,X_002,X_004,X_008,X_020,pais_nombre,pais_iso3
0,LAT2013,1,6.0,2,4,6,1,2,22,2,...,3,1,4,32,17,32301,7.0,1.239691,Argentina,ARG
1,LAT2013,2,4.0,3,4,2,1,2,22,4,...,2,1,4,32,17,32301,7.0,1.239691,Argentina,ARG
2,LAT2013,2,7.0,3,4,5,1,2,4,5,...,4,1,3,32,17,32301,7.0,0.768808,Argentina,ARG


In [64]:
df_lb.describe()

,A_003_021,D_001_061,D_001_131,S_001,S_002,S_101,S_200,S_301,X_001,X_002,X_004,X_008,X_020
count,257440.000000,347846.000000,390744.000000,489771.000000,489771.000000,469567.000000,489771.000000,489771.000000,489771.000000,489771.000000,489771.000000,400890.000000,489771.000000
mean,5.364563,2.787176,2.443840,1.515429,40.150301,3.826255,3.418024,2.308117,391.253853,1594.361218,322975.928399,5.387513,1.000079
std,3.849733,1.274505,1.039978,0.500815,16.517891,2.018585,2.227638,1.709713,269.904006,805.435588,286857.392268,2.398733,0.417132
min,-6.000000,-4.000000,-2.000000,-2.000000,-2.000000,-5.000000,-4.000000,-4.000000,32.000000,16.000000,-32635.000000,-4.000000,0.000115
25%,4.000000,2.000000,2.000000,1.000000,26.000000,2.000000,1.000000,2.000000,170.000000,1996.000000,68004.000000,4.000000,0.963109
50%,6.000000,3.000000,2.000000,2.000000,38.000000,4.000000,3.000000,3.000000,320.000000,2003.000000,218011.000000,6.000000,1.000000
75%,8.000000,3.000000,3.000000,2.000000,52.000000,5.000000,6.000000,3.000000,600.000000,2009.000000,591007.000000,7.000000,1.000000
max,99.000000,5.000000,4.000000,2.000000,100.000000,7.000000,7.000000,5.000000,862.000000,2020.000000,862033.000000,8.000000,44.022294


In [65]:
df_lb.info()

<class 'pandas.DataFrame'>
RangeIndex: 489771 entries, 0 to 489770
Data columns (total 43 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   ola            489771 non-null  str    
 1   A_001_001      488509 non-null  object 
 2   A_003_021      257440 non-null  float64
 3   A_003_031      489276 non-null  object 
 4   A_007_001      322150 non-null  object 
 5   A_007_071      468593 non-null  object 
 6   B_001_101      337473 non-null  object 
 7   B_006_061      379440 non-null  object 
 8   C_001_031      337420 non-null  object 
 9   C_003_003_011  489353 non-null  object 
 10  C_006_003_011  311784 non-null  object 
 11  D_001_001      489674 non-null  object 
 12  D_001_021      378019 non-null  object 
 13  D_001_041      377207 non-null  object 
 14  D_001_061      347846 non-null  float64
 15  D_001_091      400243 non-null  object 
 16  D_001_131      390744 non-null  float64
 17  G_002_011      340981 non-null  object 


In [67]:
import pandas as pd

resultado = []

# Recorrer todas las columnas excepto "ola"
for col in df_lb.columns:
    if col == "ola":
        continue

    conteo = (
        df_lb
        .groupby("ola")[col]
        .value_counts(dropna=False)
        .reset_index(name="total_apariciones")
    )

    conteo.insert(1, "columna", col)
    conteo.rename(columns={col: "valor"}, inplace=True)

    resultado.append(conteo)

# Unir todos los resultados
df_resultado = pd.concat(resultado, ignore_index=True)

# Exportar
df_resultado.to_csv(
    "..\data\\lb_frecuencia_valores_por_ola.csv",
    index=False,
    encoding="utf-8-sig"
)

<>:27: SyntaxWarning: invalid escape sequence '\d'
<>:27: SyntaxWarning: invalid escape sequence '\d'
C:\Users\patricio.porras\AppData\Local\Temp\ipykernel_22600\1008064725.py:27: SyntaxWarning: invalid escape sequence '\d'
  "..\data\\lb_frecuencia_valores_por_ola.csv",


LB obtener muestra

In [84]:
import pandas as pd
import numpy as np

# Semilla para que el resultado sea reproducible (opcional)
rng = np.random.default_rng(seed=42)

muestras = []

# groups es un diccionario:
# {(ola, pais_iso3): índices}
for _, idx in df_lb.groupby(["ola", "pais_iso3"]).groups.items():

    grupo = df_lb.loc[idx]

    n = min(rng.integers(18, 26), len(grupo))

    muestras.append(
        grupo.sample(
            n=n,
            random_state=rng.integers(0, 2**32 - 1)
        )
    )

df_muestra = pd.concat(muestras, ignore_index=True)

print(df_muestra.columns.tolist())   # <-- verifica aquí


# Exportar
df_muestra.to_csv(
    "..\data\lb_muestra_por_ola_pais.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Registros originales: {len(df_lb):,}")
print(f"Registros en la muestra: {len(df_muestra):,}")

<>:31: SyntaxWarning: invalid escape sequence '\d'
<>:31: SyntaxWarning: invalid escape sequence '\d'
C:\Users\patricio.porras\AppData\Local\Temp\ipykernel_22600\362027464.py:31: SyntaxWarning: invalid escape sequence '\d'
  "..\data\lb_muestra_por_ola_pais.csv",


['ola', 'A_001_001', 'A_003_021', 'A_003_031', 'A_007_001', 'A_007_071', 'B_001_101', 'B_006_061', 'C_001_031', 'C_003_003_011', 'C_006_003_011', 'D_001_001', 'D_001_021', 'D_001_041', 'D_001_061', 'D_001_091', 'D_001_131', 'G_002_011', 'G_005_001', 'H_001_011', 'H_002_011', 'H_002_031', 'H_002_041', 'H_002_101', 'H_002_111', 'H_002_131', 'H_002_161', 'H_002_241', 'I_001_001', 'S_001', 'S_002', 'S_101', 'S_200', 'S_301', 'S_700', 'S_701', 'X_001', 'X_002', 'X_004', 'X_008', 'X_020', 'pais_nombre', 'pais_iso3']
Registros originales: 489,771
Registros en la muestra: 8,921


In [83]:
df_muestra.info()

<class 'pandas.DataFrame'>
RangeIndex: 8921 entries, 0 to 8920
Data columns (total 43 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ola            8921 non-null   str    
 1   A_001_001      8899 non-null   object 
 2   A_003_021      4669 non-null   float64
 3   A_003_031      8911 non-null   object 
 4   A_007_001      5824 non-null   object 
 5   A_007_071      8521 non-null   object 
 6   B_001_101      6168 non-null   object 
 7   B_006_061      6881 non-null   object 
 8   C_001_031      6166 non-null   object 
 9   C_003_003_011  8910 non-null   object 
 10  C_006_003_011  5730 non-null   object 
 11  D_001_001      8921 non-null   object 
 12  D_001_021      6891 non-null   object 
 13  D_001_041      6843 non-null   object 
 14  D_001_061      6270 non-null   float64
 15  D_001_091      7256 non-null   object 
 16  D_001_131      7031 non-null   float64
 17  G_002_011      6140 non-null   object 
 18  G_005_001      5399